In [1]:
!pip install trl datasets transformers accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.4 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [9]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # dtype=torch.float16,
    device_map="auto"
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model parameters: 124,439,808


MMLU (Massive Multitask Language Understanding) Formal reasoning dataset - structured reasoning, not just pattern matching

In [10]:
dataset = load_dataset("lighteval/mmlu", "formal_logic", split="test")

print(dataset)
print("\nExample:")
print(dataset[0])

Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 126
})

Example:
{'question': 'Identify the conclusion of the following argument. It is hard not to verify in our peers the same weakened intelligence due to emotions that we observe in our everyday patients. The arrogance of our consciousness, which in general, belongs to the strongest defense mechanisms, blocks the unconscious complexes. Because of this, it is difficult to convince people of the unconscious, and in turn to teach them what their conscious knowledge contradicts. (Sigmund Freud, The Origin and Development of Psychoanalysis)', 'subject': 'formal_logic', 'choices': ['It is hard not to verify in our peers the same weakened intelligence due to emotions that we observe in our everyday patients.', 'The arrogance of our consciousness, which in general, belongs to the strongest defense mechanisms, blocks the unconscious complexes.', 'Because of this, it is difficult to convince people of the unc

Converts each raw MMLU example into structured prompt-response string. `SFTTrainer` expects a single text field,
`###` header is a common convention for instruction-tuned models, making the structure explicit.



In [11]:
def format_example(example):
    choices = ["A", "B", "C", "D"]
    options = "\n".join([f"{choices[i]}: {example['choices'][i]}" for i in range(len(example['choices']))])
    answer = choices[example['answer']]

    text = f"""### Question:
{example['question']}

### Options:
{options}

### Answer:
{answer}"""
    return {"text": text}

dataset = dataset.map(format_example)
print("Formatted example:")
print(dataset[0]["text"])

Formatted example:
### Question:
Identify the conclusion of the following argument. It is hard not to verify in our peers the same weakened intelligence due to emotions that we observe in our everyday patients. The arrogance of our consciousness, which in general, belongs to the strongest defense mechanisms, blocks the unconscious complexes. Because of this, it is difficult to convince people of the unconscious, and in turn to teach them what their conscious knowledge contradicts. (Sigmund Freud, The Origin and Development of Psychoanalysis)

### Options:
A: It is hard not to verify in our peers the same weakened intelligence due to emotions that we observe in our everyday patients.
B: The arrogance of our consciousness, which in general, belongs to the strongest defense mechanisms, blocks the unconscious complexes.
C: Because of this, it is difficult to convince people of the unconscious, and in turn to teach them what their conscious knowledge contradicts.
D: It is difficult to convi

## Training

- `gradient_accumulation_steps=4`instead of updating weights every batch, accumulate gradients over 4 batches, then update. This simulates a larger effective batch size (16) without needing 16 samples in GPU memory at once
- `warmup_ratio=0.1` spend first 10% of training ramping up LR up from zero before it starts decaying (prevents instability at the very start of fine-tuning)
- `max_seq_length=256` truncate any examples longer than 256 tokens, to keep memory usage predictable
- `report_to="none"` disables WandB/TensorBoard logging

In [16]:
sft_config = SFTConfig(
    output_dir="./gpt2-logic",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=False,           # change to False — GPT-2 is small enough, no need
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=10,
    max_length=256,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)

trainer.train()

/tmp/ipykernel_7715/1496196812.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


Step,Training Loss
10,3.285575


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=16, training_loss=3.4126158952713013, metrics={'train_runtime': 34.6774, 'train_samples_per_second': 7.267, 'train_steps_per_second': 0.461, 'total_flos': 22555830528000.0, 'train_loss': 3.4126158952713013, 'entropy': 4.308126052220662, 'mean_token_accuracy': 0.44440952812631923, 'num_tokens': 31838.0, 'epoch': 2.0})

In [15]:
def ask(question, choices):
    options = "\n".join([f"{c}: {o}" for c, o in zip(["A","B","C","D"], choices)])
    prompt = f"""### Question:
{question}

### Options:
{options}

### Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"Model answer: {response.strip()}")

# Test on an example from the dataset
ex = dataset[0]
ask(ex['question'], ex['choices'])
print(f"Correct answer: {['A','B','C','D'][ex['answer']]}")

Model answer: 
Correct answer: D


GPT-2 is probably too weak a base and the dataset too small to show meaningful fine-tuning results, but the pipeline is correct — the same SFTTrainer setup on a stronger base model (like Mistral-7B-Instruct) with a larger dataset should work properly.